# Getting the Scoring Sequence for each divison

In [10]:
import pandas as pd

In [11]:
series_df = pd.read_csv("../Data/mlb_division_series_results.csv")

# the CSV stores the series winner as a team code (or "TIE")
# the code below expects 1/-1/0
def get_series_result(row):
    if row["result"] == "TIE":
        return 0
    return 1 if row["result"] == row["team1"] else -1

series_df["result"] = series_df.apply(get_series_result, axis=1)

series_df = series_df[["season", "league", "division", "team1", "team2", "result", "wins1", "wins2", "games_played"]]


# GET THE SERIES WINS FOR EACH TEAM

team1 = series_df[
    ["season", "league", "division", "team1", "result"]
].copy()

team1 = team1.rename(columns={"team1": "team"})

team1["wins"] = (
    team1["result"] == 1
).astype(int)

team2 = series_df[
    ["season", "league", "division", "team2", "result"]
].copy()

team2 = team2.rename(columns={"team2": "team"})

team2["wins"] = (
    team2["result"] == -1
).astype(int)

# create a ranking
score_sequences = pd.concat(
    [team1, team2],
    ignore_index=True
)

score_sequences = (
    score_sequences
    .groupby(
        ["season", "league", "division", "team"],
        as_index=False
    )["wins"]
    .sum()
)


score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"]
)

# Getting upset frequencies per division

In [12]:
# lookup each team's score for a given season and division
score_lookup = score_sequences.set_index(
    ["season", "division", "team"]
)["wins"].to_dict()


# determine whether each series is an upset
def is_upset(row):
    if row["result"] == 0:
        return 0

    if row["result"] == 1:
        winner = row["team1"]
        loser = row["team2"]
    else:
        winner = row["team2"]
        loser = row["team1"]

    winner_score = score_lookup[
        (row["season"], row["division"], winner)
    ]

    loser_score = score_lookup[
        (row["season"], row["division"], loser)
    ]

    return int(winner_score < loser_score)


series_df["upset"] = series_df.apply(
    is_upset,
    axis=1
)

# weak upset: a tied series between teams with different scores coutns 0.5
def is_weak_upset(row):
    if row["result"] == 0:
        s1 = score_lookup[(row["season"], row["division"], row["team1"])]
        s2 = score_lookup[(row["season"], row["division"], row["team2"])]
        return 0.5 if s1 != s2 else 0
    return row["upset"]

series_df["upset_weak"] = series_df.apply(
    is_weak_upset,
    axis = 1
)

# count upsets for each season and division
upset_counts = (
    series_df
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )[["upset", "upset_weak"]]
    .sum()
)

# Merging The DFs

In [13]:
# sort teams by score within each season/division
score_sequences = score_sequences.sort_values(
    ["season", "division", "wins"],
    ascending=[True, True, False]
)

# collapse scoring sequence into one row per season/division
scoring_summary = (
    score_sequences
    .groupby(
        ["season", "league", "division"],
        as_index=False
    )
    .agg(
        num_teams=("team", "size"),
        teams=("team", tuple),
        scoring_sequence=("wins", tuple)
    )
)

# get number of in-division games played between each pair
games_per_pair = (
    series_df
    .groupby(
        ["season", "division"],
        as_index=False
    )["games_played"]
    .first()
    .rename(columns={"games_played": "games_per_pair"})
)

# merge scoring sequences, games per pair, and upset counts
analysis_df = (
    scoring_summary
    .merge(
        games_per_pair,
        on=["season", "division"],
        how="left"
    )
    .merge(
        upset_counts,
        on=["season", "league", "division"],
        how="left"
    )
)

# final column order
analysis_df = analysis_df.rename(columns={"upset": "upset_strict"})
analysis_df = analysis_df[
    [
        "season",
        "division",
        "num_teams",
        "games_per_pair",
        "teams",
        "scoring_sequence",
        "upset_strict",
        "upset_weak"
    ]
]

In [14]:
analysis_df.to_csv(
    "../Analysis/mlb_analysis.csv",
    index=False
)